# Compra Fácil Bajío · DEV de reseñas negativas

**Equipo Choch · IN1042 · Parcial II**

Josué Abraham Arteaga Manrique (18910-1) y Edgar Mauricio Trejo Delgado (192583-4).

**Estado: código preparado, sin ejecutar ni entrenar.** No hay métricas ni modelo
elegido todavía. La autoría de la preparación corresponde al commit de Josué con
apoyo de Codex; el equipo revisará y explicará lo que finalmente entregue.

El cliente quiere priorizar reseñas negativas: clase 1 = una o dos estrellas;
clase 0 = cuatro o cinco estrellas. En prueba, a umbral fijo **0.5**, pide
**exhaustividad ≥ 90 %** y **precisión ≥ 85 %**.

Se preparan una línea base trivial, **Embedding + Conv1D** y **Embedding + BiLSTM**.
Las redes se entrenarán por el equipo, sin consultas a un LLM por API.

Consultar `resenas/README.md` para instalar dependencias. Entrenar requiere activar
`ENTRENAR`; la prueba requiere activar además `EVALUAR_PRUEBA`, solo al final.


## 1. Configuración y reproducibilidad

No se ha ejecutado ninguna celda. Los límites iniciales de vocabulario, longitud y
épocas son propuestas; se pueden revisar únicamente con entrenamiento y validación.
Las semillas ayudan a reproducir el experimento, aunque dispositivo y versiones
pueden introducir diferencias numéricas. El cuaderno acepta como directorio actual
la raíz del repositorio o su carpeta `resenas/`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf
from tensorflow import keras
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score, recall_score,
    ConfusionMatrixDisplay,
)
from IPython.display import display

SEMILLA = 42
UMBRAL = 0.5
META_EXHAUSTIVIDAD = 0.90
META_PRECISION = 0.85
MAX_TOKENS = 20_000
LONGITUD = 180
DIM_EMBEDDING = 64
LOTE = 128
EPOCAS = 12
PACIENCIA = 2
TASA_APRENDIZAJE = 1e-3

# Guardas deliberadas: no entrenar ni evaluar prueba por accidente.
ENTRENAR = False
EVALUAR_PRUEBA = False

random.seed(SEMILLA)
np.random.seed(SEMILLA)
keras.utils.set_random_seed(SEMILLA)

actual = Path.cwd().resolve()
if (actual / 'resenas').is_dir():
    CARPETA = actual / 'resenas'
elif actual.name == 'resenas' and (actual / 'README.md').exists():
    CARPETA = actual
else:
    raise RuntimeError('Abrir el cuaderno desde la raíz del repositorio o resenas/.')
CARPETA_MODELO = CARPETA / 'modelo'
CARPETA_MODELO.mkdir(parents=True, exist_ok=True)
REGISTRO_PRUEBA = CARPETA_MODELO / 'evaluacion_prueba.json'
RUTA_RESULTADOS = CARPETA / 'resultados_resenas.csv'
RUTA_ERRORES = CARPETA / 'errores_resenas.csv'

VERSIONES = {
    'python': platform.python_version(), 'tensorflow': tf.__version__,
    'keras': getattr(keras, '__version__', 'incluido en tensorflow'),
    'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__,
}
CONFIGURACION = {
    'semilla': SEMILLA, 'umbral': UMBRAL, 'max_tokens': MAX_TOKENS,
    'longitud': LONGITUD, 'dimension_embedding': DIM_EMBEDDING,
    'lote': LOTE, 'epocas_maximas': EPOCAS, 'paciencia': PACIENCIA,
    'tasa_aprendizaje': TASA_APRENDIZAJE,
    'tres_estrellas': 'excluidas de entrenamiento y validacion',
}
display(pd.Series(VERSIONES, name='version'))
print('Entrenamiento habilitado:', ENTRENAR)
print('Evaluación de prueba habilitada:', EVALUAR_PRUEBA)


## 2. Carga exacta del examen y reserva de prueba

Se conserva la carga y la definición de prueba del enunciado. La existencia de
`X_prueba` y `y_prueba` no autoriza explorarlas: no se imprimen, grafican ni usan
para vocabulario, parada temprana o elección. Solo se usan en la etapa final.


In [ ]:
import pandas as pd
URL = "https://huggingface.co/datasets/SetFit/amazon_reviews_multi_es/resolve/main/{}.jsonl"
partes = {p: pd.read_json(URL.format(p), lines=True) for p in ["train", "validation", "test"]}
for df in partes.values():
    df["estrellas"] = df["label"] + 1
train, val, test = partes["train"], partes["validation"], partes["test"]
# Prueba del cliente: la partición test sin las de 3 estrellas (4,000 reseñas, mitad negativas).
# 1 = negativa. No la toquen hasta el final.
prueba = test[test["estrellas"] != 3]
X_prueba, y_prueba = prueba["text"].values, (prueba["estrellas"] <= 2).astype(int).values


## 3. Preparación y exploración sin consultar prueba

**Decisión sobre tres estrellas:** se excluyen de entrenamiento y validación porque
son neutrales y la tarea de prueba distingue únicamente negativas de positivas.

Se conservan las particiones oficiales; no se mezclan ni redistribuyen ejemplos.
Las comprobaciones no modifican silenciosamente textos o etiquetas. Los posibles
solapamientos textuales entre entrenamiento y validación se documentan como un
límite de las particiones recibidas, sin mirar prueba.


In [ ]:
for nombre, df in [('entrenamiento', train), ('validacion', val)]:
    assert {'text', 'label', 'estrellas'}.issubset(df.columns), nombre
    assert df['text'].notna().all(), f'Textos ausentes en {nombre}'
    assert df['text'].map(lambda texto: isinstance(texto, str)).all(), nombre
    assert df['estrellas'].isin([1, 2, 3, 4, 5]).all(), nombre

entrenamiento = train.loc[train['estrellas'] != 3].copy()
validacion = val.loc[val['estrellas'] != 3].copy()
for df in [entrenamiento, validacion]:
    df['negativa'] = (df['estrellas'] <= 2).astype('int32')

X_train = entrenamiento['text'].to_numpy(dtype=str)
y_train = entrenamiento['negativa'].to_numpy(dtype='int32')
X_val = validacion['text'].to_numpy(dtype=str)
y_val = validacion['negativa'].to_numpy(dtype='int32')
assert set(np.unique(y_train)) == {0, 1}
assert set(np.unique(y_val)) == {0, 1}

exploracion = pd.DataFrame([
    {
        'particion': nombre, 'filas_originales': len(original),
        'tres_estrellas_excluidas': int((original['estrellas'] == 3).sum()),
        'filas_binarias': len(binario), 'negativas': int(binario['negativa'].sum()),
        'fraccion_negativa': float(binario['negativa'].mean()),
        'textos_vacios': int(binario['text'].str.strip().eq('').sum()),
        'textos_duplicados': int(binario['text'].duplicated().sum()),
    }
    for nombre, original, binario in [
        ('entrenamiento', train, entrenamiento), ('validacion', val, validacion)
    ]
])
display(exploracion)
display(entrenamiento[['text', 'estrellas', 'negativa']].sample(6, random_state=SEMILLA))
solapamiento = len(set(X_train).intersection(set(X_val)))
print('Textos exactamente iguales compartidos entre train y validation:', solapamiento)
if solapamiento:
    print('Documentar este límite de las particiones oficiales al interpretar validación.')


In [ ]:
# Gráfica 1: balance de las clases en entrenamiento.
balance = entrenamiento['negativa'].value_counts().reindex([0, 1], fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Positiva (4-5 estrellas)', 'Negativa (1-2 estrellas)'], balance.values,
       color=['#2c7a7b', '#c05621'])
ax.set(title='Balance de clases de entrenamiento', ylabel='Número de reseñas')
for i, cantidad in enumerate(balance.values):
    ax.text(i, cantidad, f'{cantidad:,}', ha='center', va='bottom')
fig.tight_layout()
plt.show()

# Gráfica 2: longitudes de texto en entrenamiento; el vocabulario aún no se adapta.
longitudes = entrenamiento['text'].str.split().str.len()
fig, ax = plt.subplots(figsize=(7, 4))
for etiqueta, nombre, color in [(0, 'Positiva', '#2c7a7b'), (1, 'Negativa', '#c05621')]:
    ax.hist(longitudes[entrenamiento['negativa'] == etiqueta], bins=50,
            alpha=0.55, label=nombre, color=color)
ax.axvline(LONGITUD, color='black', linestyle='--', label=f'Límite inicial: {LONGITUD}')
ax.set(title='Longitud de las reseñas de entrenamiento',
       xlabel='Palabras separadas por espacios', ylabel='Número de reseñas')
ax.legend()
fig.tight_layout()
plt.show()
display(longitudes.describe(percentiles=[0.50, 0.90, 0.95, 0.99]).to_frame('palabras'))


## 4. Vocabulario aprendido solo con entrenamiento

`TextVectorization` convierte a minúsculas y elimina puntuación; no se eliminan
acentos ni palabras de negación. Se usan tokens enteros, con 0 reservado para
relleno y 1 para desconocidos. El vocabulario se adapta únicamente a `X_train`.
Ambas redes reciben la misma preparación. Se cuantifica el truncado y la tasa de
tokens desconocidos sin consultar prueba.


In [ ]:
vectorizador = keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize='lower_and_strip_punctuation',
    split='whitespace', output_mode='int', output_sequence_length=LONGITUD,
    name='vectorizador_entrenamiento',
)
vectorizador.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(LOTE))
TAM_VOCABULARIO = len(vectorizador.get_vocabulary())
print('Tamaño del vocabulario, incluidos relleno y desconocidos:', TAM_VOCABULARIO)

# Mismo vocabulario, sin longitud fija, solo para medir cuántos tokens se perderían.
medidor = keras.layers.TextVectorization(
    standardize='lower_and_strip_punctuation', split='whitespace',
    output_mode='int', ragged=True, vocabulary=vectorizador.get_vocabulary(),
)

def diagnostico_textos(textos):
    total_tokens = desconocidos = truncadas = total_filas = 0
    for lote_textos in tf.data.Dataset.from_tensor_slices(textos).batch(LOTE):
        tokens = medidor(lote_textos)
        tamanos = tokens.row_lengths().numpy()
        total_filas += len(tamanos)
        truncadas += int(np.sum(tamanos > LONGITUD))
        valores = tokens.flat_values.numpy()
        total_tokens += len(valores)
        desconocidos += int(np.sum(valores == 1))
    return {
        'fraccion_truncada': truncadas / max(total_filas, 1),
        'fraccion_tokens_desconocidos': desconocidos / max(total_tokens, 1),
    }

diagnosticos = pd.DataFrame({
    'entrenamiento': diagnostico_textos(X_train),
    'validacion': diagnostico_textos(X_val),
}).T
display(diagnosticos)

def crear_datos(textos, etiquetas=None, entrenar=False):
    # Vectorizar por lote en CPU y almacenar enteros int32 para reutilizar las épocas.
    if etiquetas is None:
        datos = tf.data.Dataset.from_tensor_slices(textos).batch(LOTE)
        return datos.map(lambda x: tf.cast(vectorizador(x), tf.int32),
                         num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    datos = tf.data.Dataset.from_tensor_slices((textos, etiquetas)).batch(LOTE)
    datos = datos.map(lambda x, y: (tf.cast(vectorizador(x), tf.int32), y),
                      num_parallel_calls=tf.data.AUTOTUNE).unbatch().cache()
    if entrenar:
        datos = datos.shuffle(min(len(textos), 20_000), seed=SEMILLA,
                              reshuffle_each_iteration=True)
    return datos.batch(LOTE).prefetch(tf.data.AUTOTUNE)

datos_train = crear_datos(X_train, y_train, entrenar=True)
datos_val = crear_datos(X_val, y_val)
datos_val_prediccion = crear_datos(X_val)


## 5. Métricas y línea base trivial

Exhaustividad = VP/(VP+FN); precisión = VP/(VP+FP). F1 sirve únicamente para
selección y análisis de validación. La exactitud se reporta porque la pide el examen.
Las etiquetas siempre se obtienen con probabilidad ≥ 0.5; no se ajusta el umbral.
La base predice la clase mayoritaria aprendida en entrenamiento y tiene cero
parámetros entrenables de red.


In [ ]:
def metricas_cliente(y_real, probabilidades):
    probabilidades = np.asarray(probabilidades, dtype=float).reshape(-1)
    y_real = np.asarray(y_real).reshape(-1)
    if len(y_real) != len(probabilidades) or len(y_real) == 0:
        raise ValueError('Las etiquetas y puntuaciones deben tener igual longitud no vacía.')
    if not np.isfinite(probabilidades).all() or not ((probabilidades >= 0) & (probabilidades <= 1)).all():
        raise ValueError('Se requieren probabilidades finitas entre 0 y 1.')
    if not np.isin(y_real, [0, 1]).all():
        raise ValueError('Las etiquetas deben ser binarias: 1 = negativa.')
    etiquetas = (probabilidades >= UMBRAL).astype(int)
    exhaustividad = float(recall_score(y_real, etiquetas, zero_division=0))
    precision = float(precision_score(y_real, etiquetas, zero_division=0))
    return {
        'exhaustividad': exhaustividad,
        'precision': precision,
        'exactitud': float(accuracy_score(y_real, etiquetas)),
        'f1': float(f1_score(y_real, etiquetas, zero_division=0)),
        'cumple': bool(exhaustividad >= META_EXHAUSTIVIDAD and precision >= META_PRECISION),
    }

def parametros_entrenables(modelo):
    return int(sum(np.prod(peso.shape) for peso in modelo.trainable_weights))

def ordenar_candidatos(tabla):
    # Solo redes; la base es referencia. No recibe datos ni métricas de prueba.
    candidatas = tabla.loc[tabla['tipo'] != 'linea base'].copy()
    if candidatas.empty:
        raise ValueError('Faltan modelos del parcial para elegir.')
    candidatas['cobertura_objetivos'] = np.minimum(
        candidatas['exhaustividad'] / META_EXHAUSTIVIDAD,
        candidatas['precision'] / META_PRECISION,
    )
    return candidatas.sort_values(
        ['cumple', 'cobertura_objetivos', 'f1', 'parametros', 'modelo'],
        ascending=[False, False, False, True, True],
    ).reset_index(drop=True)

linea_base = DummyClassifier(strategy='most_frequent', random_state=SEMILLA)
linea_base.fit(np.zeros((len(y_train), 1)), y_train)
indice_negativa = list(linea_base.classes_).index(1)
prob_base_val = linea_base.predict_proba(np.zeros((len(y_val), 1)))[:, indice_negativa]
fila_base_val = {
    'modelo': 'Clase mayoritaria', 'tipo': 'linea base', 'parametros': 0,
    **metricas_cliente(y_val, prob_base_val),
}
display(pd.DataFrame([fila_base_val]))


## 6. Dos modelos del parcial, todavía sin entrenar

**Conv1D:** aprende patrones locales de palabras con embeddings entrenables,
convolución y agregación máxima. No usa máscara de relleno: ese efecto se deberá
revisar al comparar longitudes y errores.

**BiLSTM:** aprende contexto en ambos sentidos con embeddings entrenables y máscara
para ignorar tokens de relleno. Su costo puede ser mayor, especialmente en CPU.

Cada arquitectura tiene su propio embedding inicializado desde cero. No se utiliza
conocimiento preentrenado. La parada temprana observa `val_loss`, nunca prueba.


In [ ]:
def construir_conv1d():
    entrada = keras.Input(shape=(LONGITUD,), dtype='int32', name='tokens')
    x = keras.layers.Embedding(TAM_VOCABULARIO, DIM_EMBEDDING, name='embedding')(entrada)
    x = keras.layers.Conv1D(64, 5, padding='same', activation='relu')(x)
    x = keras.layers.GlobalMaxPooling1D()(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    salida = keras.layers.Dense(1, activation='sigmoid', name='probabilidad_negativa')(x)
    return keras.Model(entrada, salida, name='embedding_conv1d')

def construir_bilstm():
    entrada = keras.Input(shape=(LONGITUD,), dtype='int32', name='tokens')
    x = keras.layers.Embedding(TAM_VOCABULARIO, DIM_EMBEDDING,
                               mask_zero=True, name='embedding')(entrada)
    x = keras.layers.Bidirectional(keras.layers.LSTM(32))(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    salida = keras.layers.Dense(1, activation='sigmoid', name='probabilidad_negativa')(x)
    return keras.Model(entrada, salida, name='embedding_bilstm')

CONSTRUCTORES = {
    'Embedding + Conv1D': ('convolucional', construir_conv1d),
    'Embedding + BiLSTM': ('recurrente', construir_bilstm),
}


## 7. Entrenamiento y comparación exclusivamente en validación

**Esta celda se detiene antes de entrenar con la configuración inicial.** Para
entrenar, activar `ENTRENAR` y ejecutar en orden desde el inicio. Si existe un
registro de prueba, se bloquea un nuevo desarrollo en esta carpeta para evitar
ajustes posteriores contra resultados ya vistos.


In [ ]:
if not ENTRENAR:
    raise RuntimeError('Entrenamiento desactivado. Cambiar ENTRENAR a True solo cuando se autorice entrenar.')
if REGISTRO_PRUEBA.exists():
    raise RuntimeError('Ya existe registro de prueba. No volver a entrenar o elegir usando esa prueba.')

modelos = {}
historiales = {}
probabilidades_val = {'Clase mayoritaria': prob_base_val}
filas_val = [fila_base_val]

for nombre, (tipo, constructor) in CONSTRUCTORES.items():
    keras.utils.set_random_seed(SEMILLA)
    modelo = constructor()
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=TASA_APRENDIZAJE),
        loss='binary_crossentropy',
        metrics=[keras.metrics.BinaryAccuracy(name='exactitud', threshold=UMBRAL)],
    )
    modelo.summary()
    historial = modelo.fit(
        datos_train, validation_data=datos_val, epochs=EPOCAS,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=PACIENCIA, restore_best_weights=True,
        )], verbose=1,
    )
    prob_val = modelo.predict(datos_val_prediccion, verbose=0).reshape(-1)
    modelos[nombre] = modelo
    historiales[nombre] = {k: [float(v) for v in valores]
                           for k, valores in historial.history.items()}
    probabilidades_val[nombre] = prob_val
    filas_val.append({
        'modelo': nombre, 'tipo': tipo, 'parametros': parametros_entrenables(modelo),
        **metricas_cliente(y_val, prob_val),
    })

resultados_validacion = pd.DataFrame(filas_val)
clasificacion_validacion = ordenar_candidatos(resultados_validacion)
NOMBRE_ELEGIDO = str(clasificacion_validacion.iloc[0]['modelo'])
resultados_validacion['elegido'] = resultados_validacion['modelo'].eq(NOMBRE_ELEGIDO)
display(resultados_validacion)
print('Elección fijada únicamente con validación:', NOMBRE_ELEGIDO)
if not bool(clasificacion_validacion.iloc[0]['cumple']):
    print('Ninguna red cumple ambas metas en validación. No prometer cumplimiento al cliente.')

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nombre, historial) in zip(ejes, historiales.items()):
    ax.plot(np.arange(1, len(historial['loss']) + 1), historial['loss'], label='Entrenamiento')
    ax.plot(np.arange(1, len(historial['val_loss']) + 1), historial['val_loss'], label='Validación')
    ax.set(title=nombre, xlabel='Época', ylabel='Entropía cruzada')
    ax.legend()
fig.tight_layout()
plt.show()


## 8. Guardado del modelo elegido antes de consultar prueba

El artefacto `.keras` recibe textos crudos e integra la preparación y la red elegida.
Se comprueba su recarga sobre ocho textos de validación, sin usar prueba.
Se registra la regla de elección y la comparación de validación para justificarla.
El CSV de validación se guarda en `modelo/` y no sustituye al CSV de prueba exigido.


In [ ]:
entrada_texto = keras.Input(shape=(), dtype=tf.string, name='resena')
salida = modelos[NOMBRE_ELEGIDO](vectorizador(entrada_texto))
modelo_elegido = keras.Model(entrada_texto, salida, name='clasificador_resenas')
RUTA_MODELO = CARPETA_MODELO / 'modelo_resenas.keras'
modelo_elegido.save(RUTA_MODELO)

recargado = keras.models.load_model(RUTA_MODELO, compile=False)
sonda = tf.constant(X_val[:8].tolist(), dtype=tf.string)
antes = modelo_elegido(sonda, training=False).numpy()
despues = recargado(sonda, training=False).numpy()
np.testing.assert_allclose(antes, despues, rtol=1e-5, atol=1e-6)
print('Recarga comprobada con textos de validación:', RUTA_MODELO.name)
peso_mb = RUTA_MODELO.stat().st_size / 1_000_000
print(f'Tamaño del modelo: {peso_mb:.2f} MB')
if peso_mb > 100:
    print('Supera 100 MB: alojar en Hugging Face o Drive y agregar enlace público al README.')

resultados_validacion.to_csv(CARPETA_MODELO / 'resultados_validacion.csv', index=False)
exploracion.to_csv(CARPETA_MODELO / 'exploracion_desarrollo.csv', index=False)
diagnosticos.to_csv(CARPETA_MODELO / 'diagnosticos_texto.csv', index_label='particion')
seleccion = {
    'modelo_elegido': NOMBRE_ELEGIDO,
    'origen_eleccion': 'validation oficial sin tres estrellas',
    'criterio': 'cumplimiento, minimo de cocientes de metas, F1, menos parametros, nombre',
    'configuracion': CONFIGURACION, 'versiones': VERSIONES,
    'resultados_validacion': json.loads(resultados_validacion.to_json(orient='records')),
    'historiales': historiales,
    'solapamiento_textual_train_val': solapamiento,
    'creado_utc': datetime.now(timezone.utc).isoformat(),
}
(CARPETA_MODELO / 'seleccion_validacion.json').write_text(
    json.dumps(seleccion, ensure_ascii=False, indent=2), encoding='utf-8',
)


## 9. Etapa final de prueba: una sola evaluación de candidatos congelados

Mantener `EVALUAR_PRUEBA=False` hasta cerrar todas las decisiones. La elección ya
está fijada y no cambia aunque otra red obtenga mejores métricas en prueba.
El registro se crea de forma exclusiva antes de predecir; si ya terminó, se leen
los CSV guardados en lugar de volver a evaluar. Si se interrumpe, revisar lo ocurrido
con el equipo antes de cualquier acción; no borrar el registro para seguir ajustando.

`cumple` exige ambas metas, con valores completos sin redondear. `elegido` identifica
exactamente una red. No hay un límite de parámetros en este problema.


In [ ]:
if not EVALUAR_PRUEBA:
    print('Prueba reservada. No se han calculado métricas de prueba.')
elif REGISTRO_PRUEBA.exists():
    registro = json.loads(REGISTRO_PRUEBA.read_text(encoding='utf-8'))
    if registro.get('estado') != 'completada':
        raise RuntimeError('Hay una evaluación de prueba iniciada o interrumpida. Revisar sin volver a evaluar.')
    resultados = pd.read_csv(RUTA_RESULTADOS)
    errores = pd.read_csv(RUTA_ERRORES)
    print('Se muestran resultados guardados; no se volvió a predecir sobre prueba.')
    display(resultados)
    display(errores)
else:
    assert len(y_prueba) == 4_000 and int(np.sum(y_prueba)) == 2_000
    assert NOMBRE_ELEGIDO in modelos
    assert RUTA_MODELO.exists()
    registro = {
        'estado': 'iniciada', 'modelo_elegido': NOMBRE_ELEGIDO,
        'umbral': UMBRAL, 'inicio_utc': datetime.now(timezone.utc).isoformat(),
        'candidatos': ['Clase mayoritaria', *modelos.keys()],
    }
    # 'x' falla si otra sesión creó el registro: no sobrescribir ni repetir.
    with REGISTRO_PRUEBA.open('x', encoding='utf-8') as archivo:
        json.dump(registro, archivo, ensure_ascii=False, indent=2)

    datos_prueba = crear_datos(X_prueba.astype(str))
    probabilidades_prueba = {
        'Clase mayoritaria': linea_base.predict_proba(
            np.zeros((len(y_prueba), 1))
        )[:, indice_negativa],
    }
    for nombre, modelo in modelos.items():
        probabilidades_prueba[nombre] = modelo.predict(datos_prueba, verbose=0).reshape(-1)

    filas = []
    for fila in filas_val:
        nombre = fila['modelo']
        medidas = metricas_cliente(y_prueba, probabilidades_prueba[nombre])
        filas.append({
            'modelo': nombre, 'tipo': fila['tipo'], 'parametros': fila['parametros'],
            'exhaustividad': medidas['exhaustividad'], 'precision': medidas['precision'],
            'exactitud': medidas['exactitud'], 'cumple': medidas['cumple'],
            'elegido': nombre == NOMBRE_ELEGIDO,
        })
    resultados = pd.DataFrame(filas, columns=[
        'modelo', 'tipo', 'parametros', 'exhaustividad', 'precision',
        'exactitud', 'cumple', 'elegido',
    ])
    assert resultados['elegido'].sum() == 1
    resultados.to_csv(RUTA_RESULTADOS, index=False)

    probabilidades_elegido = probabilidades_prueba[NOMBRE_ELEGIDO]
    etiquetas_elegido = (probabilidades_elegido >= UMBRAL).astype(int)
    indices_error = np.flatnonzero(etiquetas_elegido != y_prueba)
    # Tomar primero un falso negativo y un falso positivo, cuando existan;
    # completar con errores en orden original, sin volver a inferir.
    seleccion_errores = []
    for clase_real in [1, 0]:
        grupo = indices_error[y_prueba[indices_error] == clase_real]
        if len(grupo):
            seleccion_errores.append(int(grupo[0]))
    for indice in indices_error:
        if int(indice) not in seleccion_errores:
            seleccion_errores.append(int(indice))
        if len(seleccion_errores) >= 3:
            break
    seleccion_errores = seleccion_errores[:3]
    errores = pd.DataFrame([
        {
            'indice_prueba': i, 'texto': X_prueba[i],
            'estrellas': int(prueba.iloc[i]['estrellas']),
            'clase_real': int(y_prueba[i]), 'clase_predicha': int(etiquetas_elegido[i]),
            'probabilidad_negativa': float(probabilidades_elegido[i]),
            'tipo_error': 'falso negativo' if y_prueba[i] == 1 else 'falso positivo',
            'analisis': 'PENDIENTE: explicar evidencia textual, causa posible e impacto.',
        }
        for i in seleccion_errores
    ], columns=[
        'indice_prueba', 'texto', 'estrellas', 'clase_real', 'clase_predicha',
        'probabilidad_negativa', 'tipo_error', 'analisis',
    ])
    errores.to_csv(RUTA_ERRORES, index=False)
    matriz = confusion_matrix(y_prueba, etiquetas_elegido, labels=[0, 1])
    registro.update({
        'estado': 'completada', 'fin_utc': datetime.now(timezone.utc).isoformat(),
        'total_errores_elegido': int(len(indices_error)),
        'matriz_confusion_orden_0_1': matriz.tolist(),
    })
    REGISTRO_PRUEBA.write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding='utf-8')

    display(resultados)
    display(errores)
    ConfusionMatrixDisplay(matriz, display_labels=['Positiva', 'Negativa']).plot(cmap='Blues')
    plt.title(f'Prueba final: {NOMBRE_ELEGIDO}')
    plt.show()
    elegida = resultados.loc[resultados['elegido']].iloc[0]
    print(f"De cada 100 reseñas negativas, detecta aproximadamente {100 * elegida['exhaustividad']:.1f}.")
    print(f"De cada 100 marcadas como negativas, aproximadamente {100 * elegida['precision']:.1f} lo son.")
    print('Cumple ambas especificaciones:', bool(elegida['cumple']))
    print('Errores totales del modelo elegido:', len(indices_error))
    if len(indices_error) < 3:
        print('Hay menos de tres errores reales: documentarlo y consultar al docente; no inventarlos.')


## 10. Interpretación que el equipo completará con evidencia real

**Pendiente: todavía no existen resultados.** No convertir estas notas en
afirmaciones sobre desempeño antes de ejecutar y revisar el DEV.

### Justificación del modelo elegido

Completar con el nombre fijado en validación, sus métricas y parámetros, la diferencia
frente a la otra red y la base, y su relación con el costo operativo del cliente.
La regla automática de selección se definió antes de mirar prueba; no cambiarla
después. Si no cumple las metas en prueba, reconocerlo y explicar mejoras por
validar con datos nuevos.

### Tres errores del modelo elegido

Usar las filas reales de `errores_resenas.csv`. Por cada una, anotar:

1. Texto, estrellas, etiqueta real, predicción y probabilidad.
2. Evidencia concreta del texto que permita interpretar el fallo.
3. Causa posible, identificada como hipótesis si no se ha comprobado.
4. Impacto para atención a clientes y mejora propuesta.

Un falso negativo deja sin prioridad una queja; un falso positivo ocupa capacidad
de atención. La selección incluye ambos tipos cuando existen. Completar los tres
análisis en este cuaderno y en el CSV; no basta con imprimir los ejemplos.

### Limitaciones para respaldar en el folleto

- **Cobertura geográfica:** Amazon España frente a una tienda mexicana. Citar la
  procedencia y un error relevante si existe; no afirmar una caída no medida.
- **Reseñas neutrales:** el experimento excluye tres estrellas. Reportar cuántas
  filas se excluyeron en el desarrollo y delimitar el uso previsto.
- **Truncado y vocabulario:** incorporar las fracciones calculadas en desarrollo y
  relacionarlas con errores reales si se observa ese patrón.
- **Etiquetas por estrellas:** examinar si algún fallo contradice la calificación;
  no suponer que cualquier desacuerdo del modelo es ruido de etiqueta.

### Aviso ético y mejora

Propuesta: usar el modelo para apoyar la priorización, conservar revisión humana y
evitar que una predicción descarte definitivamente la atención a una persona. No
se ha medido equidad por grupos. Validar con reseñas mexicanas representativas y
revisar falsos negativos antes de desplegarlo.

### Cierre de la entrega

Guardar el cuaderno con sus salidas, actualizar el README general con el modelo
elegido, verificar `resultados_resenas.csv`, completar los tres análisis y elaborar
`folleto_resenas.pdf` de máximo dos cuartillas con maqueta y cifras coincidentes.
El PMV funcional es opcional y no se implementa en esta preparación.

## Referencias de implementación

- [TextVectorization de Keras](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/).
- [Guardado y serialización de Keras](https://www.tensorflow.org/guide/keras/serialization_and_saving).
- Fuente, particiones y criterios: enunciado del examen IN1042, Parcial II, caso aplicado.
